In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# Create folders
# ==========================================

os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv('../data/credit_card_data.csv')

print("Dataset Loaded")
print(df.shape)

# ==========================================
# Features for Clustering
# ==========================================

CLUSTER_FEATURES = [

    'monthly_spend',

    'travel_spend',

    'shopping_spend',

    'food_spend',

    'txn_count',

    'avg_txn_value',

    'util_rate',

    'reward_pts_mo',

    'intl_txn_pct'
]

X = df[CLUSTER_FEATURES].fillna(0)

# ==========================================
# Scaling
# ==========================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ==========================================
# Elbow Method
# ==========================================

print()
print("Running Elbow Method...")

inertias = []
sil_scores = []

K_range = range(2, 8)

for k in K_range:

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    km.fit(X_scaled)

    inertias.append(km.inertia_)

    sil = silhouette_score(
        X_scaled,
        km.labels_,
        sample_size=3000
    )

    sil_scores.append(sil)

    print(
        f"K={k} | "
        f"Inertia={km.inertia_:.0f} | "
        f"Silhouette={sil:.4f}"
    )

# ==========================================
# Plot Elbow + Silhouette
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

# Elbow

axes[0].plot(
    list(K_range),
    inertias,
    marker='o',
    linewidth=2
)

axes[0].set_title(
    'Elbow Method',
    fontweight='bold'
)

axes[0].set_xlabel('K')

axes[0].set_ylabel('Inertia')

# Silhouette

axes[1].plot(
    list(K_range),
    sil_scores,
    marker='o',
    color='green',
    linewidth=2
)

axes[1].set_title(
    'Silhouette Score',
    fontweight='bold'
)

axes[1].set_xlabel('K')

axes[1].set_ylabel('Score')

plt.tight_layout()

plt.savefig(
    '../plots/kmeans_analysis.png',
    dpi=100,
    bbox_inches='tight'
)

plt.close()

print()
print("Saved:")
print("plots/kmeans_analysis.png")

# ==========================================
# Final KMeans Model
# ==========================================

print()
print("Training Final KMeans...")

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=15,
    max_iter=300
)

df['cluster'] = kmeans.fit_predict(X_scaled)

final_sil = silhouette_score(
    X_scaled,
    df['cluster'],
    sample_size=5000
)

print(
    f"Final Silhouette Score: {final_sil:.4f}"
)

print()
print("Cluster Distribution:")

print(
    df['cluster'].value_counts()
)

# ==========================================
# Persona Mapping
# ==========================================

cluster_summary = df.groupby('cluster').agg({

    'monthly_spend': 'mean',

    'travel_spend': 'mean',

    'shopping_spend': 'mean',

    'util_rate': 'mean',

    'txn_count': 'mean',

    'intl_txn_pct': 'mean'
}).round(2)

print()
print("Cluster Summary:")
print(cluster_summary)

# ==========================================
# Persona Logic
# ==========================================

persona_map = {}

for cluster_id, row in cluster_summary.iterrows():

    if (
        row['travel_spend'] >
        row['shopping_spend']
    ) and (
        row['intl_txn_pct'] > 0.10
    ):

        persona_map[cluster_id] = 'Traveller'

    elif (
        row['shopping_spend'] > 25000
    ):

        persona_map[cluster_id] = 'Premium Shopper'

    elif (
        row['util_rate'] > 0.70
    ):

        persona_map[cluster_id] = 'At-Risk'

    elif (
        row['monthly_spend'] < 20000
    ):

        persona_map[cluster_id] = 'Minimalist'

    else:

        persona_map[cluster_id] = 'Everyday User'

df['persona'] = df['cluster'].map(
    persona_map
)

# ==========================================
# Persona Distribution
# ==========================================

print()
print("Persona Distribution:")

print(
    df['persona'].value_counts()
)

# ==========================================
# Persona Chart
# ==========================================

plt.figure(figsize=(8,5))

persona_counts = df['persona'].value_counts()

plt.bar(
    persona_counts.index,
    persona_counts.values,
    color=[
        '#2563EB',
        '#7C3AED',
        '#16A34A',
        '#F97316',
        '#EF4444'
    ]
)

plt.title(
    'Customer Persona Distribution',
    fontweight='bold'
)

plt.ylabel('Customers')

plt.xticks(rotation=15)

plt.tight_layout()

plt.savefig(
    '../plots/persona_distribution.png',
    dpi=100,
    bbox_inches='tight'
)

plt.close()

# ==========================================
# Save Models
# ==========================================

joblib.dump(
    kmeans,
    '../models/kmeans_model.pkl'
)

joblib.dump(
    scaler,
    '../models/cluster_scaler.pkl'
)

json.dump(
    {
        'cluster_features': CLUSTER_FEATURES,
        'silhouette_score': round(final_sil, 4),
        'persona_map': {
            str(k): v for k, v in persona_map.items()
        }
    },
    open('../models/cluster_metrics.json', 'w'),
    indent=2
)

# ==========================================
# Save Dataset
# ==========================================

df.to_csv(
    '../data/cc_segmented.csv',
    index=False
)

print()
print("Saved:")
print("models/kmeans_model.pkl")
print("models/cluster_scaler.pkl")
print("models/cluster_metrics.json")
print("data/cc_segmented.csv")

Dataset Loaded
(30000, 22)

Running Elbow Method...
K=2 | Inertia=185212 | Silhouette=0.3683
K=3 | Inertia=157818 | Silhouette=0.2229
K=4 | Inertia=142987 | Silhouette=0.1711
K=5 | Inertia=130113 | Silhouette=0.1782
K=6 | Inertia=121011 | Silhouette=0.1528
K=7 | Inertia=113775 | Silhouette=0.1496

Saved:
plots/kmeans_analysis.png

Training Final KMeans...
Final Silhouette Score: 0.1783

Cluster Distribution:
cluster
1    10957
0     9989
3     4891
4     3183
2      980
Name: count, dtype: int64

Cluster Summary:
         monthly_spend  travel_spend  shopping_spend  util_rate  txn_count  \
cluster                                                                      
0             51690.20       7558.28        12569.49       0.66      16.17   
1             24669.47       3621.58         6077.56       0.32      16.16   
2            223789.62      36531.93        58032.16       0.74      15.31   
3            113648.44      17202.52        28820.99       0.68      15.64   
4            